In [39]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import json
import scipy
import tqdm
from copy import deepcopy
import anndata as ad
import SDP_miRNA.dataset
import SDP_miRNA.optimization
import SDP_miRNA.optimization_MOSEK
import SDP_miRNA.correlation
import SDP_miRNA.optimization_condition
import SDP_miRNA.constraints

# Cell Cycle

In [40]:
# load pcRNA
adata_pcRNA = ad.read_h5ad("../HEK293T-Cell-Cycle/Subsampling-data/TotalX_HEK293T_SS_pcRNA.h5ad")

# load miRNA
adata_miRNA = ad.read_h5ad("../HEK293T-Cell-Cycle/Subsampling-data/TotalX_HEK293T_SS_miRNA.h5ad")

# load capture
beta = np.loadtxt("../HEK293T/TotalX_HEK293T_capture.txt")
beta_G1 = np.loadtxt("../HEK293T-Cell-Cycle/TotalX_HEK293T_G1_capture.txt")
beta_S = np.loadtxt("../HEK293T-Cell-Cycle/TotalX_HEK293T_S_capture.txt")
beta_G2M = np.loadtxt("../HEK293T-Cell-Cycle/TotalX_HEK293T_G2M_capture.txt")

# max observed count per miRNA
max_count = adata_miRNA.X.max(axis=0).toarray()

# select
count_mask = (max_count >= 3).squeeze()
miRNA_idxs = np.arange(len(count_mask))[count_mask]

# subset miRNA
adata_miRNA = adata_miRNA[:, miRNA_idxs]

# display
print(f"{sum(count_mask)} / {len(count_mask)} miRNA retained")

20 / 41 miRNA retained


In [75]:
def cell_cycle_pipeline(d, c, G):

    # select miRNA and pcRNA
    miRNA = adata_miRNA.var['GeneName'].iloc[0]
    pcRNA = adata_pcRNA.var['GeneName'].iloc[:G]

    # initialize datasets
    SDP_data_G1 = SDP_miRNA.dataset.Dataset()
    SDP_data_S = SDP_miRNA.dataset.Dataset()
    SDP_data_G2M = SDP_miRNA.dataset.Dataset()

    # construct datasets
    SDP_data_G1.construct_dataset_adata(
        adata_miRNA[adata_miRNA.obs['phase'] == "G1", adata_miRNA.var['GeneName'] == miRNA],
        adata_pcRNA[adata_pcRNA.obs['phase'] == "G1", adata_pcRNA.var['GeneName'].isin(pcRNA)],
        beta_G1
    )
    SDP_data_S.construct_dataset_adata(
        adata_miRNA[adata_miRNA.obs['phase'] == "S", adata_miRNA.var['GeneName'] == miRNA],
        adata_pcRNA[adata_pcRNA.obs['phase'] == "S", adata_pcRNA.var['GeneName'].isin(pcRNA)],
        beta_S
    )
    SDP_data_G2M.construct_dataset_adata(
        adata_miRNA[adata_miRNA.obs['phase'] == "G2M", adata_miRNA.var['GeneName'] == miRNA],
        adata_pcRNA[adata_pcRNA.obs['phase'] == "G2M", adata_pcRNA.var['GeneName'].isin(pcRNA)],
        beta_G2M
    )

    # bootstrap
    SDP_data_G1.bootstrap(d=d, confidence=c)
    SDP_data_S.bootstrap(d=d, confidence=c)
    SDP_data_G2M.bootstrap(d=d, confidence=c)

    # constraints
    constraints = SDP_miRNA.constraints.Constraint(
        moment_bounds=True,
        moment_matrices=True,
        factorization=False
    )

    # consistency of G1, S, G2M
    MF_condition_G1_S_G2M = SDP_miRNA.optimization_condition.OptimizationCondition(
        [SDP_data_G1, SDP_data_S, SDP_data_G2M],
        constraints,
        reactions=[],
        vrs=np.array([]),
        db=0,
        R=0,
        S=2,
        U=[],
        d=d
    )
    MF_condition_G1_S_G2M.optimized_construction = True
    MF_condition_G1_S_G2M.analyse_dataset()

    # consistency of S, G2M
    MF_condition_S_G2M = SDP_miRNA.optimization_condition.OptimizationCondition(
        [SDP_data_S, SDP_data_G2M],
        constraints,
        reactions=[],
        vrs=np.array([]),
        db=0,
        R=0,
        S=2,
        U=[],
        d=d
    )
    MF_condition_S_G2M.optimized_construction = True
    MF_condition_S_G2M.analyse_dataset()

    # consistency of G1, G2M
    MF_condition_G1_G2M = SDP_miRNA.optimization_condition.OptimizationCondition(
        [SDP_data_G1, SDP_data_G2M],
        constraints,
        reactions=[],
        vrs=np.array([]),
        db=0,
        R=0,
        S=2,
        U=[],
        d=d
    )
    MF_condition_G1_G2M.optimized_construction = True
    MF_condition_G1_G2M.analyse_dataset()

    # consistency of G1, S
    MF_condition_G1_S = SDP_miRNA.optimization_condition.OptimizationCondition(
        [SDP_data_G1, SDP_data_S],
        constraints,
        reactions=[],
        vrs=np.array([]),
        db=0,
        R=0,
        S=2,
        U=[],
        d=d
    )
    MF_condition_G1_S.optimized_construction = True
    MF_condition_G1_S.analyse_dataset()

    # extract status
    status_G1_S_G2M = [dct['status'] for dct in MF_condition_G1_S_G2M.result_dict.values()]
    status_S_G2M = [dct['status'] for dct in MF_condition_S_G2M.result_dict.values()]
    status_G1_G2M = [dct['status'] for dct in MF_condition_G1_G2M.result_dict.values()]
    status_G1_S = [dct['status'] for dct in MF_condition_G1_S.result_dict.values()]

    # construct table
    status_table = pd.DataFrame(
        np.array([status_G1_S_G2M, status_S_G2M, status_G1_G2M, status_G1_S]).T,
        columns=["G1_S_G2M", "S_G2M", "G1_G2M", "G1_S"],
        index=pcRNA
    )

    return status_table

In [76]:
# settings
d = 3
c = 0.95
G = 1000

status_table_cc_95 = cell_cycle_pipeline(d, c, G)

100%|██████████| 1000/1000 [00:03<00:00, 269.44it/s]


In [77]:
status_table_cc_95.value_counts()

G1_S_G2M    S_G2M       G1_G2M      G1_S      
INFEASIBLE  INFEASIBLE  INFEASIBLE  INFEASIBLE    389
            OPTIMAL     INFEASIBLE  INFEASIBLE    345
OPTIMAL     OPTIMAL     OPTIMAL     OPTIMAL       123
INFEASIBLE  INFEASIBLE  OPTIMAL     OPTIMAL        63
                        INFEASIBLE  OPTIMAL        34
                        OPTIMAL     INFEASIBLE     24
            OPTIMAL     INFEASIBLE  OPTIMAL        14
                        OPTIMAL     INFEASIBLE      8
Name: count, dtype: int64

G1_S_G2M    S_G2M       G1_G2M      G1_S      
INFEASIBLE  INFEASIBLE  INFEASIBLE  INFEASIBLE    4099
            OPTIMAL     INFEASIBLE  INFEASIBLE    3642
OPTIMAL     OPTIMAL     OPTIMAL     OPTIMAL        995
INFEASIBLE  INFEASIBLE  OPTIMAL     OPTIMAL        616
                        INFEASIBLE  OPTIMAL        443
                        OPTIMAL     INFEASIBLE     247
            OPTIMAL     INFEASIBLE  OPTIMAL        143
                        OPTIMAL     INFEASIBLE     119
                                    OPTIMAL          3
Name: count, dtype: int64

# Subsampling Control

In [78]:
# Loaded Subsampling adata, as contains cell cycle and additional subsampling 

# load capture
beta_N1 = np.loadtxt("../HEK293T-Cell-Cycle/Subsampling-data/TotalX_HEK293T_SS1_capture.txt")
beta_N2 = np.loadtxt("../HEK293T-Cell-Cycle/Subsampling-data/TotalX_HEK293T_SS2_capture.txt")

In [79]:
def subsampling_pipeline(d, c, G):

    # select miRNA and pcRNA
    miRNA = adata_miRNA.var['GeneName'].iloc[0]
    pcRNA = adata_pcRNA.var['GeneName'].iloc[:G]

    # initialize datasets
    SDP_data_N1 = SDP_miRNA.dataset.Dataset()
    SDP_data_N2 = SDP_miRNA.dataset.Dataset()

    # construct datasets
    SDP_data_N1.construct_dataset_adata(
        adata_miRNA[adata_miRNA.obs['Subsample_1'], adata_miRNA.var['GeneName'] == miRNA],
        adata_pcRNA[adata_pcRNA.obs['Subsample_1'], adata_pcRNA.var['GeneName'].isin(pcRNA)],
        beta_N1
    )
    SDP_data_N2.construct_dataset_adata(
        adata_miRNA[adata_miRNA.obs['Subsample_2'], adata_miRNA.var['GeneName'] == miRNA],
        adata_pcRNA[adata_pcRNA.obs['Subsample_2'], adata_pcRNA.var['GeneName'].isin(pcRNA)],
        beta_N2
    )

    # bootstrap
    SDP_data_N1.bootstrap(d=d, confidence=c)
    SDP_data_N2.bootstrap(d=d, confidence=c)

    # constraints
    constraints = SDP_miRNA.constraints.Constraint(
        moment_bounds=True,
        moment_matrices=True,
        factorization=False
    )

    # consistency of N1, N2
    MF_condition_N1_N2 = SDP_miRNA.optimization_condition.OptimizationCondition(
        [SDP_data_N1, SDP_data_N2],
        constraints,
        reactions=[],
        vrs=np.array([]),
        db=0,
        R=0,
        S=2,
        U=[],
        d=d
    )
    MF_condition_N1_N2.optimized_construction = True
    MF_condition_N1_N2.analyse_dataset()

    # extract status
    status_N1_N2 = [dct['status'] for dct in MF_condition_N1_N2.result_dict.values()]

    return status_N1_N2

In [80]:
# settings
d = 3
c = 0.95
G = 1000

status_ss_95 = subsampling_pipeline(d, c, G)

100%|██████████| 1000/1000 [00:04<00:00, 216.58it/s]


In [85]:
# consistenct of N1 - N2
print("N1 - N2")
print(np.unique(status_ss_95, return_counts=True))

# compare to same size change with cell cycle
print("G1 - S")
print(np.unique(status_table_cc_95['G1_S'].values.tolist(), return_counts=True))
print("G1 - G2M")
print(np.unique(status_table_cc_95['G1_G2M'].values.tolist(), return_counts=True))

N1 - N2
(array(['INFEASIBLE', 'OPTIMAL'], dtype='<U10'), array([108, 892]))
G1 - S
(array(['INFEASIBLE', 'OPTIMAL'], dtype='<U10'), array([766, 234]))
G1 - G2M
(array(['INFEASIBLE', 'OPTIMAL'], dtype='<U10'), array([782, 218]))


Splitting by cell cycle clearly leads to significantly more infeasibility, but it is worrying how many pairs are infeasible under just subsampling cells

# Confidence Level

In [86]:
# settings
d = 3
c = 0.99
G = 1000

status_table_cc_99 = cell_cycle_pipeline(d, c, G)
status_ss_99 = subsampling_pipeline(d, c, G)

100%|██████████| 1000/1000 [00:04<00:00, 240.85it/s]


In [88]:
status_table_cc_99.value_counts()

G1_S_G2M    S_G2M       G1_G2M      G1_S      
INFEASIBLE  OPTIMAL     INFEASIBLE  INFEASIBLE    392
            INFEASIBLE  INFEASIBLE  INFEASIBLE    253
OPTIMAL     OPTIMAL     OPTIMAL     OPTIMAL       190
INFEASIBLE  INFEASIBLE  OPTIMAL     OPTIMAL        74
                        INFEASIBLE  OPTIMAL        41
            OPTIMAL     INFEASIBLE  OPTIMAL        22
                        OPTIMAL     INFEASIBLE     16
            INFEASIBLE  OPTIMAL     INFEASIBLE     12
Name: count, dtype: int64

In [87]:
# consistenct of N1 - N2
print("N1 - N2")
print(np.unique(status_ss_99, return_counts=True))

# compare to same size change with cell cycle
print("G1 - S")
print(np.unique(status_table_cc_99['G1_S'].values.tolist(), return_counts=True))
print("G1 - G2M")
print(np.unique(status_table_cc_99['G1_G2M'].values.tolist(), return_counts=True))

N1 - N2
(array(['INFEASIBLE', 'OPTIMAL'], dtype='<U10'), array([ 33, 967]))
G1 - S
(array(['INFEASIBLE', 'OPTIMAL'], dtype='<U10'), array([673, 327]))
G1 - G2M
(array(['INFEASIBLE', 'OPTIMAL'], dtype='<U10'), array([708, 292]))


Changing to 99% confidence intervals we see subsampling drop to much less infeasibility, while cell cycle remains very high

Suggests bootstrap intervals may be too tight, leading to some false positives.

# Alternative Bootstrap

# Appendix

In [22]:
# consistency of single phase check
MF_condition_test = SDP_miRNA.optimization_condition.OptimizationCondition(
    [SDP_data_G2M],
    constraints,
    reactions=[],
    vrs=np.array([]),
    db=0,
    R=0,
    S=2,
    U=[],
    d=d
)
MF_condition_test.analyse_dataset()
np.unique([dct['status'] for dct in MF_condition_test.result_dict.values()], return_counts=True)

100%|██████████| 100/100 [00:01<00:00, 50.66it/s]


(array(['OPTIMAL'], dtype='<U7'), array([100]))

In [ ]:
import plotly.express as px

In [ ]:
px.parallel_categories(status_table)